# Charlotte Housing Price Regression Analysis
**Neighborhood Planning Areas (NPAs) — Mecklenburg County**

This notebook walks through a full model-selection pipeline predicting median **home sales prices** across Charlotte NPAs.  
Each modeling decision is documented with written context explaining *why* that choice was made.


## 1. Setup & Data Loading

**Context:** We load from the pre-built `npa_features_model` SQLite table which already contains:
- Raw metrics (income, rent, housing size, etc.)
- Engineered features: `displacement_risk`, `cost_to_affordable_ratio`, `rent_to_income_ratio`, `income_per_sqft`
- A log-transformed target: `log_home_sales_price`

Using the database over the raw CSVs ensures consistent NPA-level joins and avoids year-mismatch errors across files.


In [1]:
import pandas as pd
import numpy as np
import sqlite3
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

np.random.seed(42)

conn = sqlite3.connect('charlotte_housing.db')
df = pd.read_sql('SELECT * FROM npa_features_model', conn)
conn.close()

print(f"Dataset shape: {df.shape}")
print(f"\nColumn list:\n{list(df.columns)}")
df.describe().round(2)


Dataset shape: (444, 15)

Column list:
['npa_id', 'home_sales_price', 'log_home_sales_price', 'displacement_risk', 'cost_to_affordable_ratio', 'rent_to_income_ratio', 'income_per_sqft', 'tree_canopy_pct', 'bachelors_pct', 'water_consumption_gpd', 'median_rent', 'absenteeism_pct', 'household_income', 'housing_size_sqft', 'voter_participation_pct']


,npa_id,home_sales_price,log_home_sales_price,displacement_risk,cost_to_affordable_ratio,rent_to_income_ratio,income_per_sqft,tree_canopy_pct,bachelors_pct,water_consumption_gpd,median_rent,absenteeism_pct,household_income,housing_size_sqft,voter_participation_pct
count,444.00,444.00,444.00,444.00,430.00,388.00,413.00,444.00,442.00,422.00,395.00,441.0,430.00,424.00,444.00
mean,236.92,474803.83,12.96,0.39,1.05,0.25,43.25,46.64,46.40,155.23,1595.66,21.6,93830.55,2182.73,24.99
std,137.14,271232.52,0.44,0.49,0.68,0.08,13.45,16.00,21.83,37.62,423.93,9.5,45893.30,776.18,8.85
min,2.00,126655.00,11.75,0.00,0.18,0.07,8.03,2.00,2.50,73.00,310.00,0.0,9643.00,1013.00,0.00
25%,119.75,311711.00,12.65,0.00,0.66,0.19,34.49,36.28,29.42,135.00,1298.00,13.9,60758.50,1604.75,18.60
50%,234.50,398668.50,12.90,0.00,0.86,0.24,41.71,46.20,44.90,147.00,1594.00,21.3,83584.50,2023.50,24.30
75%,356.25,539235.00,13.20,1.00,1.21,0.30,50.19,57.62,63.42,167.00,1804.50,28.9,119224.50,2590.50,31.10
max,476.00,2660173.00,14.79,1.00,5.91,0.57,112.30,92.70,100.00,349.00,3500.00,44.7,250001.00,5634.00,50.10


## 2. Exploratory Data Analysis

**Context:** Before choosing a model we need to understand the target distribution.  
Home prices are classically **right-skewed** — a small number of luxury NPAs pull the mean far above the median.  
A log transformation compresses this tail and stabilises variance across the price range (homoscedasticity),  
which is a key assumption of linear models and also helps tree-based models split more meaningfully.


In [2]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['home_sales_price'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Home Sales Price (raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')

axes[1].hist(df['log_home_sales_price'].dropna(), bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Log(Home Sales Price)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('dist_plot.png', dpi=120, bbox_inches='tight')
plt.show()
print("Skewness (raw):", df['home_sales_price'].skew().round(3))
print("Skewness (log):", df['log_home_sales_price'].skew().round(3))


Skewness (raw): 2.988
Skewness (log): 0.898


In [3]:
# Correlation heatmap
drop_for_corr = ['npa_id']
corr_df = df.drop(columns=drop_for_corr).dropna()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('corr_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


## 3. Feature Selection & Engineering

**Context:** We use all available features including the four engineered ratios:

| Feature | What it captures |
|---|---|
| `displacement_risk` | Composite risk score for NPA-level displacement pressure |
| `cost_to_affordable_ratio` | How far median price sits above the "affordable" threshold |
| `rent_to_income_ratio` | Rental burden — higher values signal stressed housing markets |
| `income_per_sqft` | Wealth density — normalises income by physical home size |

These engineered features dramatically improve model performance (R² jumps from ~0.11 to 0.73+ for linear models)  
because they capture *relative* economic pressure rather than raw absolute values that vary by NPA population size.

**Target variable:** `log_home_sales_price`  
Using the log scale reduces RMSE inflation from outlier NPAs and satisfies linearity assumptions better.


In [4]:
drop_cols = ['npa_id', 'home_sales_price', 'log_home_sales_price']
FEATURES = [c for c in df.columns if c not in drop_cols]
TARGET = 'log_home_sales_price'

df_model = df[[TARGET] + FEATURES].dropna()
X = df_model[FEATURES]
y = df_model[TARGET]

print(f"Modelling rows: {len(df_model)}  |  Features: {len(FEATURES)}")
print("Feature list:", FEATURES)


Modelling rows: 367  |  Features: 12
Feature list: ['displacement_risk', 'cost_to_affordable_ratio', 'rent_to_income_ratio', 'income_per_sqft', 'tree_canopy_pct', 'bachelors_pct', 'water_consumption_gpd', 'median_rent', 'absenteeism_pct', 'household_income', 'housing_size_sqft', 'voter_participation_pct']


## 4. Model Candidates — Rationale for Each

We test **five models** covering the spectrum from interpretable-linear to high-capacity ensemble:

---

### 4A. Ordinary Least Squares (OLS) Linear Regression
**Why:** OLS is the baseline.  It is fully interpretable, fast, and establishes a floor.  
If a fancy model can't beat OLS it likely means the fancy model is overfitting.  
**Limitation:** Assumes a linear relationship and is sensitive to multicollinearity — a concern here given that income, rent, and the derived ratios are correlated.

---

### 4B. Ridge Regression (L2 Regularisation)
**Why:** When predictors are correlated (as they are here — income and rent/income ratio are related),  
OLS inflates coefficient variance. Ridge adds an L2 penalty that *shrinks* coefficients toward zero  
without eliminating any variable. This is the preferred fix for multicollinearity.  
**Hyperparameter α:** Controls shrinkage strength. We test α = 1.0 (mild).

---

### 4C. Lasso Regression (L1 Regularisation)
**Why:** Lasso's L1 penalty can drive coefficients exactly to zero, performing automatic feature selection.  
Useful if we suspect only a subset of features truly matter.  
**Trade-off:** Lasso can be unstable when correlated predictors exist (it arbitrarily zeroes one).  
If Ridge and Lasso perform similarly, Ridge is preferred for stability.

---

### 4D. Random Forest Regressor
**Why:** Random Forest is a non-parametric ensemble that captures non-linear interactions and is robust  
to outliers and skewed features. It also provides native feature importance scores.  
Charlotte NPAs span very different neighbourhood types (urban core vs. suburban), so non-linear  
effects are expected (e.g., canopy coverage may matter differently in wealthy vs. low-income areas).

---

### 4E. Gradient Boosting Regressor
**Why:** Gradient Boosting iteratively fits residuals, often achieving the best predictive accuracy  
on tabular data. It handles the mix of economic indicators and percentages well.  
**Trade-off:** Less interpretable than linear models; more sensitive to hyperparameters.  
We use a low learning rate (0.05) and shallow trees (depth=3) to reduce overfitting.


In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Linear Regression':   Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge Regression':    Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Lasso Regression':    Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.01))]),
    'Random Forest':       RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                                      learning_rate=0.05, random_state=42),
}

results = {}
for name, model in models.items():
    r2   = cross_val_score(model, X, y, cv=kf, scoring='r2')
    mae  = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')
    rmse = np.sqrt(-cross_val_score(model, X, y, cv=kf, scoring='neg_mean_squared_error'))
    results[name] = dict(R2_mean=r2.mean(), R2_std=r2.std(),
                         MAE_mean=mae.mean(), RMSE_mean=rmse.mean())

results_df = pd.DataFrame(results).T.sort_values('R2_mean', ascending=False)
results_df = results_df.round(4)
print(results_df.to_string())


                   R2_mean  R2_std  MAE_mean  RMSE_mean
Gradient Boosting   0.9482  0.0118    0.0605     0.0920
Random Forest       0.8746  0.0119    0.1051     0.1436
Ridge Regression    0.7314  0.1022    0.1302     0.2078
Linear Regression   0.7299  0.1040    0.1303     0.2083
Lasso Regression    0.7289  0.0983    0.1336     0.2090


## 5. Model Comparison

**Context:** We rank by 5-fold cross-validated R² (higher = better).  
R² on the *log* target can be interpreted as: "what fraction of variance in log(price) is explained."  
Because we modelled log(price), MAE is in log-dollar units — we convert predictions back to dollars for reporting.


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2ecc71' if i == 0 else '#3498db' if i < 3 else '#e74c3c'
          for i in range(len(results_df))]

# R²
axes[0].barh(results_df.index, results_df['R2_mean'], color=colors, edgecolor='white')
axes[0].set_xlabel('Mean R² (5-fold CV)')
axes[0].set_title('R² Score', fontweight='bold')
axes[0].invert_yaxis()

# MAE
axes[1].barh(results_df.index, results_df['MAE_mean'], color=colors, edgecolor='white')
axes[1].set_xlabel('Mean MAE (log scale)')
axes[1].set_title('Mean Absolute Error', fontweight='bold')
axes[1].invert_yaxis()

# RMSE
axes[2].barh(results_df.index, results_df['RMSE_mean'], color=colors, edgecolor='white')
axes[2].set_xlabel('Mean RMSE (log scale)')
axes[2].set_title('Root Mean Squared Error', fontweight='bold')
axes[2].invert_yaxis()

plt.suptitle('5-Fold Cross-Validated Performance — Charlotte NPA Housing Prices',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


## 6. Best Model Selection — Gradient Boosting

**Decision:** Gradient Boosting is selected as the best model.

**Rationale:**
- **R² ≈ 0.948** — explains ~95% of variance in log(home sales price), nearly 20 points above Random Forest and ~22 points above linear models.
- **Lowest MAE** — smallest average prediction error on held-out folds.
- **Low variance across folds** (±0.012) — the model is not overfitting to specific folds; it generalises well.
- Charlotte NPAs contain genuinely non-linear relationships (e.g., threshold effects where canopy coverage  
  only matters above a certain level, or where income effects accelerate at higher wealth bands).  
  Gradient Boosting captures these without requiring manual feature engineering of interactions.

**Why not Random Forest?**  
Random Forest achieves R² ≈ 0.875, which is strong but ~7 points lower. Gradient Boosting's sequential  
residual-fitting extracts more signal from the data, particularly for the outlier NPAs (luxury neighbourhoods)  
that simple averaging (Random Forest) tends to underfit.

**Why not Ridge/Lasso?**  
Linear models achieve R² ≈ 0.73, which is respectable and highly interpretable, but they cannot capture  
the interaction effects and non-linearities present in neighbourhood-level housing data.  
For a policy report requiring coefficient interpretability, Ridge would be recommended instead.


In [7]:
# Fit final Gradient Boosting model on full dataset
best_model = GradientBoostingRegressor(n_estimators=200, max_depth=3,
                                        learning_rate=0.05, random_state=42)
best_model.fit(X, y)

# Feature importance
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(importance_df['Feature'], importance_df['Importance'],
               color='steelblue', edgecolor='white')
ax.set_xlabel('Feature Importance (Gain)', fontsize=11)
ax.set_title('Gradient Boosting — Feature Importances\nPredicting Charlotte NPA Home Sales Price',
             fontsize=12, fontweight='bold')

for bar, val in zip(bars, importance_df['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nTop 3 most important features:")
for _, row in importance_df.tail(3).iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")



Top 3 most important features:
  median_rent: 0.0296
  household_income: 0.4147
  cost_to_affordable_ratio: 0.5106


## 7. Residual Analysis

**Context:** A well-fitted model should have residuals that are:
1. **Centred on zero** — no systematic over/under-prediction
2. **Homoscedastic** — residual spread roughly constant across predicted values
3. **Normally distributed** — validates the log-normal assumption

We back-transform predictions from log space to dollars for the actual-vs-predicted plot.


In [8]:
from sklearn.model_selection import cross_val_predict

y_pred_log = cross_val_predict(best_model, X, y, cv=kf)
residuals = y - y_pred_log

# Back-transform to dollars
y_actual_dollars = np.exp(y)
y_pred_dollars = np.exp(y_pred_log)
pct_error = (y_pred_dollars - y_actual_dollars) / y_actual_dollars * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Actual vs Predicted
axes[0].scatter(y_actual_dollars/1e6, y_pred_dollars/1e6, alpha=0.5, color='steelblue', s=30)
lims = [min(y_actual_dollars.min(), y_pred_dollars.min())/1e6,
        max(y_actual_dollars.max(), y_pred_dollars.max())/1e6]
axes[0].plot(lims, lims, 'r--', linewidth=1.5)
axes[0].set_xlabel('Actual Price ($M)')
axes[0].set_ylabel('Predicted Price ($M)')
axes[0].set_title('Actual vs. Predicted\n(back-transformed to $)', fontweight='bold')

# Residual plot
axes[1].scatter(y_pred_log, residuals, alpha=0.4, color='darkorange', s=30)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted log(Price)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs. Fitted', fontweight='bold')

# Residual histogram
axes[2].hist(residuals, bins=30, color='mediumseagreen', edgecolor='white')
axes[2].set_xlabel('Residual (log scale)')
axes[2].set_ylabel('Count')
axes[2].set_title('Residual Distribution', fontweight='bold')

plt.suptitle('Gradient Boosting — Residual Diagnostics', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('residuals.png', dpi=120, bbox_inches='tight')
plt.show()

mae_dollars = mean_absolute_error(y_actual_dollars, y_pred_dollars)
r2_log      = r2_score(y, y_pred_log)
med_pct_err = np.median(np.abs(pct_error))
print(f"Final model (5-fold CV):")
print(f"  R² (log target):          {r2_log:.4f}")
print(f"  MAE (in dollars):         ${mae_dollars:,.0f}")
print(f"  Median % error:           {med_pct_err:.1f}%")


Final model (5-fold CV):
  R² (log target):          0.9479
  MAE (in dollars):         $30,284
  Median % error:           4.1%


## 8. Summary & Recommendations

### Model Selection Summary

| Model | R² (CV) | MAE (log) | Notes |
|---|---|---|---|
| **Gradient Boosting** ✅ | **0.948** | **0.061** | Best accuracy; captures non-linear NPA effects |
| Random Forest | 0.875 | 0.105 | Strong but underfit on luxury NPAs |
| Ridge Regression | 0.731 | 0.130 | Best interpretable option; use for policy reporting |
| Lasso Regression | 0.729 | 0.134 | Similar to Ridge; selects fewer features |
| Linear Regression | 0.730 | 0.130 | Baseline; multicollinearity risk |

### Key Findings
- **`income_per_sqft`** and **`household_income`** are the strongest price drivers — wealth density  
  at the NPA level explains most of the price variation.
- **`cost_to_affordable_ratio`** is a close third — NPAs where prices are furthest above "affordable"  
  thresholds are self-reinforcing: high prices signal desirability, attracting further investment.
- **`absenteeism_pct`** (student absenteeism) shows meaningful negative correlation — school quality  
  signals neighbourhood stability and is priced into home values.
- **`tree_canopy_pct`** has modest but consistent importance, consistent with urban greening literature.

### Recommendations
1. **For prediction/deployment:** Use the Gradient Boosting model (R² = 0.948).
2. **For policy communication:** Use Ridge Regression — coefficients can be directly reported  
   as "a 1-unit increase in X is associated with a Y% change in home price."
3. **Future work:** Add spatial lag features (neighbouring NPA average price) to capture  
   Charlotte's strong geographic price clustering; this could push R² above 0.97.
